# Sleep-EDF: comprobar dos archivos reales en Colab

Este notebook valida **solo dos archivos** con `SleepEDFDataset`, sin modificar señales ni arquitectura. No entrena un modelo.

Necesitas este notebook y el archivo `protopnet/eeg_utilities/sleep_edf_dataset.py` creado en la etapa anterior. Si el repositorio no está en Colab, la celda 3 te permitirá subir únicamente ese archivo Python desde tu equipo. No se instala el antiguo `requirements.txt`: para esta comprobación bastan NumPy, PyTorch, pandas y matplotlib, disponibles en Colab.

Ejecuta las celdas en orden. Las salidas se generarán con tus datos; aquí no se incluyen resultados reales.

## 1. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Elegir exactamente dos archivos

Se seleccionan los dos primeros por nombre. Para comprobar dos noches conocidas de una misma persona, escribe sus nombres en `FILE_NAMES`. La selección no depende de cargar señales. Ajusta la ruta si tu carpeta está en otra ubicación de Drive.

In [ ]:
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/SleepCassette_procesado')
FILE_NAMES = []  # Ejemplo: ['noche_1.npz', 'noche_2.npz']

assert DATA_DIR.is_dir(), f'No existe la carpeta: {DATA_DIR}'
if FILE_NAMES:
    assert len(FILE_NAMES) == 2 and len(set(FILE_NAMES)) == 2
    assert all(Path(name).name == name and name.endswith('.npz') for name in FILE_NAMES)
    selected_files = sorted(DATA_DIR / name for name in FILE_NAMES)
else:
    selected_files = sorted(DATA_DIR.glob('*.npz'))[:2]
assert len(selected_files) == 2, 'Se necesitan dos archivos .npz distintos'
assert all(path.is_file() for path in selected_files)
print('Archivos seleccionados:', [path.name for path in selected_files])


## 3. Importar el cargador implementado

Si tienes el repositorio en Colab, ajusta `LOADER_PATH`. Si no existe esa ruta, sube **solo `sleep_edf_dataset.py`**, no tus datos privados. Se utiliza la implementación de la etapa anterior, sin duplicarla ni cambiarla.

In [ ]:
import importlib.util

LOADER_PATH = Path('/content/ProtoEEG/protopnet/eeg_utilities/sleep_edf_dataset.py')
if not LOADER_PATH.is_file():
    from google.colab import files
    print('Selecciona sleep_edf_dataset.py de la etapa anterior')
    uploaded = files.upload()
    assert set(uploaded) == {'sleep_edf_dataset.py'}, 'Sube únicamente sleep_edf_dataset.py'
    LOADER_PATH = Path('/content/sleep_edf_dataset.py')
    LOADER_PATH.write_bytes(uploaded['sleep_edf_dataset.py'])
    del uploaded
spec = importlib.util.spec_from_file_location('sleep_edf_dataset', LOADER_PATH)
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
SleepEDFDataset = module.SleepEDFDataset


## 4. Inspeccionar cada archivo por separado

Se libera `X` después de revisar cada archivo. `subj` puede ser un identificador escalar o un vector por época. No se habilita pickle: si tus archivos guardan `subj` como objetos, se deberá revisar ese formato antes de continuar.

El orden Fpz-Cz, Pz-Oz, EOG horizontal y los 100 Hz deben proceder de tu preprocesamiento; las dimensiones por sí solas no los demuestran.

In [ ]:
import gc
import numpy as np
import pandas as pd
from IPython.display import display

records = []
expected_labels = []
expected_subjects = []
for path in selected_files:
    with np.load(path, allow_pickle=False) as data:
        assert {'X', 'y', 'subj'} <= set(data.files), f'Faltan claves en {path.name}'
        x, y, subj = data['X'], data['y'], data['subj']
    assert x.ndim == 3 and x.shape[1:] == (3, 3000) and len(x) > 0
    assert y.shape == (len(x),) and np.issubdtype(y.dtype, np.integer)
    assert np.isin(y, np.arange(1, 7)).all()
    assert np.isfinite(x).all()
    assert subj.dtype.kind in 'iuUS', 'subj debe contener enteros o cadenas'
    if subj.size == 1:
        subjects = np.repeat(subj.reshape(-1), len(x)).astype(str)
    else:
        assert subj.shape == (len(x),), 'subj debe ser escalar o vector (N,)'
        subjects = subj.astype(str)
    assert all(s.strip() for s in subjects)
    ids = sorted(set(subjects.tolist()))
    print(f'\n{path.name}: X={x.shape}, tipo={x.dtype}, y={y.shape}, subj={subj.shape}')
    print('Etiquetas originales:', np.unique(y), '| Primeras etiquetas:', y[:20])
    print('Sujetos:', ids)
    records.append({'archivo': path.name, 'epocas': len(x), 'sujetos': ids})
    expected_labels.append(y.astype(np.int64) - 1)
    expected_subjects.extend(subjects.tolist())
    del x, y, subj, subjects
    gc.collect()
expected_labels = np.concatenate(expected_labels)
display(pd.DataFrame(records))


## 5. Cargar únicamente esos dos archivos

El cargador existente lee todos los `.npz` de la carpeta indicada. Para limitarlo sin modificarlo, copiamos únicamente los dos elegidos a un directorio temporal local. No se cambia ningún archivo de Drive. Necesitarás espacio local para esas dos copias y RAM para sus señales, incluyendo la concatenación temporal.

In [ ]:
import shutil
import tempfile

with tempfile.TemporaryDirectory(prefix='sleep_edf_two_') as staging:
    for path in selected_files:
        shutil.copy2(path, Path(staging) / path.name)
    dataset = SleepEDFDataset(staging)
# El dataset conserva las señales en RAM; las copias temporales ya se eliminaron.
assert len(dataset) == sum(row['epocas'] for row in records)
np.testing.assert_array_equal(dataset.y.numpy(), expected_labels)
assert dataset.subject_ids == expected_subjects
assert len(set(dataset.sample_ids)) == len(dataset)
print('Conversión exacta verificada: y_nueva = y_original - 1')
print('Mapeo:', dict(enumerate(dataset.class_names)))
print('Primeras etiquetas convertidas:', dataset.y[:20].tolist())
print(f'Señales retenidas: {dataset.x.numel() * dataset.x.element_size() / 1e9:.3f} GB')


## 6. Distribución de las seis clases en los dos archivos

Se muestran también clases ausentes, con conteo cero. Esta distribución no representa necesariamente todo Sleep-EDF.

In [ ]:
import matplotlib.pyplot as plt

counts = np.bincount(dataset.y.numpy(), minlength=6)
distribution = pd.DataFrame({
    'etiqueta_original': np.arange(1, 7),
    'etiqueta_modelo': np.arange(6),
    'clase': dataset.class_names,
    'epocas': counts,
    'porcentaje': counts / len(dataset) * 100,
})
assert counts.sum() == len(dataset)
display(distribution)
plt.bar(dataset.class_names, counts)
plt.ylabel('Número de épocas')
plt.title('Distribución de clases: dos archivos seleccionados')
plt.show()


## 7. Obtener un lote con DataLoader

In [ ]:
import torch
from torch.utils.data import DataLoader

loader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)
batch = next(iter(loader))
assert batch['img'].shape == (min(16, len(dataset)), 3, 3000)
assert batch['img'].dtype == torch.float32
assert batch['target'].shape == (len(batch['img']),)
assert batch['target'].dtype == torch.int64
assert torch.isfinite(batch['img']).all()
assert ((batch['target'] >= 0) & (batch['target'] <= 5)).all()
print('Señales:', batch['img'].shape, batch['img'].dtype)
print('Etiquetas:', batch['target'].tolist())
print('Sujetos:', batch['subject_id'])
print('IDs de época:', batch['sample_id'][:3])


## 8. Comprobar agrupación por sujeto

Esta celda agrupa las épocas y archivos por el `subj` real y comprueba que una partición por sujetos mantiene cada grupo unido. La asignación ilustrativa a dos grupos no es una división de entrenamiento definitiva; con dos archivos no siempre se pueden formar tres conjuntos.

**Límite de la comprobación:** si dos noches conocidas de una misma persona tienen IDs diferentes, agrupar por ID no detectará el error. Contrasta los identificadores impresos con tus metadatos. No se debe inferir identidad a partir del nombre del archivo sin conocer su convención.

In [ ]:
from collections import defaultdict

files_by_subject = defaultdict(set)
for row in records:
    for subject in row['sujetos']:
        files_by_subject[subject].add(row['archivo'])
display(pd.DataFrame([
    {'sujeto': subject, 'archivos': sorted(names),
     'epocas': dataset.subject_ids.count(subject)}
    for subject, names in sorted(files_by_subject.items())
]))
shared = [s for s, names in files_by_subject.items() if len(names) == 2]
print('IDs presentes en ambos archivos:', shared)

# Opcional: indica aquí el ID si sabes que elegiste dos noches de la misma persona.
EXPECTED_SAME_SUBJECT = None  # Ejemplo: 'SC400'
if EXPECTED_SAME_SUBJECT is not None:
    assert all(set(row['sujetos']) == {str(EXPECTED_SAME_SUBJECT)} for row in records), (
        'Los IDs no coinciden con el sujeto esperado: revisa tus metadatos'
    )

subjects = sorted(files_by_subject)
group_by_subject = {subject: i % 2 for i, subject in enumerate(subjects)}
epoch_groups = np.array([group_by_subject[s] for s in dataset.subject_ids])
subject_array = np.array(dataset.subject_ids)
for subject in subjects:
    assert len(set(epoch_groups[subject_array == subject])) == 1
left = set(subject_array[epoch_groups == 0])
right = set(subject_array[epoch_groups == 1])
assert left.isdisjoint(right)
print('Agrupación sin fuga de sujetos verificada:', group_by_subject)
if not shared:
    print('No hay IDs compartidos: no se ha comprobado un caso real de dos noches de una persona.')
if len(subjects) == 1:
    print('Solo hay un sujeto: sus épocas deben permanecer juntas; el segundo grupo queda vacío.')


## 9. Cómo evitar cargar todo el conjunto al entrenar

No ampliar este cargador en RAM a toda la carpeta. El siguiente paso propuesto para datos, sin cambiar la arquitectura, es:

1. Construir un índice pequeño con archivo, número de épocas, etiquetas y sujeto; separar sujetos antes de crear los conjuntos.
2. Convertir una vez cada `X` de `.npz` a un `.npy` local y mantener etiquetas/metadatos por separado. Leer cada archivo secuencialmente durante esa conversión.
3. Crear un Dataset que use `np.load(..., mmap_mode='r')` sobre esos `.npy` y copie únicamente la época solicitada al tensor. Las señales se leen bajo demanda; no se concatenan todos los registros.
4. Mantener un lote pequeño y empezar con `num_workers=0`. Usar almacenamiento local de Colab para las lecturas aleatorias, conservando los originales en Drive.

`mmap_mode='r'` no permite acceder de esa manera a las matrices dentro de un `.npz`, incluso si se indica en `np.load`: hay que extraer/convertir primero. Reducir `batch_size` tampoco elimina la carga completa que hace el Dataset actual.

Esta estrategia queda propuesta, **no implementada en esta etapa**. Los archivos reales y su identidad de sujeto se validan al ejecutar este notebook en tu Colab; las pruebas locales usan datos sintéticos. El lote `(B, 3, 3000)` todavía no está conectado al backbone original.